In [3]:
import torch
from transformers import pipeline, BertTokenizer, BertModel

# ==========================================
# PHẦN 1: MASKED LANGUAGE MODELING (MLM)
# ==========================================
print("--- 1. TÁC VỤ MASKED LANGUAGE MODELING (MLM) ---")
# Khởi tạo pipeline cho tác vụ điền từ, sử dụng BERT cơ bản
mlm_pipeline = pipeline("fill-mask", model="bert-base-uncased")

# Câu văn bị che mất một từ bằng token [MASK]
text_mlm = "The water of Walden Pond is so beautifully [MASK]."
results = mlm_pipeline(text_mlm)

print(f"Câu đầu vào: {text_mlm}\n")
for res in results[:3]: # In ra 3 dự đoán có xác suất cao nhất
    print(f"Dự đoán: {res['token_str']:<10} | Xác suất: {res['score']:.4f} | Câu hoàn chỉnh: {res['sequence']}")

--- 1. TÁC VỤ MASKED LANGUAGE MODELING (MLM) ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Câu đầu vào: The water of Walden Pond is so beautifully [MASK].

Dự đoán: beautiful  | Xác suất: 0.1503 | Câu hoàn chỉnh: the water of walden pond is so beautifully beautiful.
Dự đoán: flowing    | Xác suất: 0.0841 | Câu hoàn chỉnh: the water of walden pond is so beautifully flowing.
Dự đoán: preserved  | Xác suất: 0.0609 | Câu hoàn chỉnh: the water of walden pond is so beautifully preserved.


In [4]:
# ==========================================
# PHẦN 2: NAMED ENTITY RECOGNITION (NER)
# ==========================================
print("\n--- 2. TÁC VỤ NHẬN DIỆN THỰC THỂ (NER) ---")
# Sử dụng một mô hình BERT đã được fine-tune sẵn cho bài toán NER
# aggregation_strategy="simple" giúp gộp các sub-word (do tokenization) thành từ hoàn chỉnh
ner_pipeline = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")

# Câu ví dụ trích từ tài liệu (Figure 10.12)
text_ner = "Jane Villanueva of United Airlines Holding discussed the Chicago route."
entities = ner_pipeline(text_ner)

print(f"Câu đầu vào: {text_ner}\n")
for ent in entities:
    # In ra thực thể và nhãn tương ứng (PER, ORG, LOC)
    print(f"Thực thể: {ent['word']:<25} | Nhãn: {ent['entity_group']:<5} | Độ tin cậy: {ent['score']:.4f}")



--- 2. TÁC VỤ NHẬN DIỆN THỰC THỂ (NER) ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Câu đầu vào: Jane Villanueva of United Airlines Holding discussed the Chicago route.

Thực thể: Jane Villanueva           | Nhãn: PER   | Độ tin cậy: 0.9893
Thực thể: United Airlines Holding   | Nhãn: ORG   | Độ tin cậy: 0.9991
Thực thể: Chicago                   | Nhãn: LOC   | Độ tin cậy: 0.9995


In [5]:
# ==========================================
# PHẦN 3: TRÍCH XUẤT CONTEXTUAL EMBEDDINGS
# ==========================================
print("\n--- 3. TRÍCH XUẤT CONTEXTUAL EMBEDDINGS ---")
# Khởi tạo tokenizer và kiến trúc mô hình cơ sở
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Hai câu có chứa từ "bank" nhưng mang ý nghĩa khác nhau (đa nghĩa)
text_1 = "I need to deposit money in the bank." # Ngân hàng
text_2 = "We sat by the river bank." # Bờ sông

# Tiền xử lý văn bản thành các tensor ID
inputs_1 = tokenizer(text_1, return_tensors="pt")
inputs_2 = tokenizer(text_2, return_tensors="pt")

# Đưa dữ liệu qua mô hình để lấy embedding
with torch.no_grad():
    outputs_1 = model(**inputs_1)
    outputs_2 = model(**inputs_2)

# Lấy tensor đầu ra ở lớp cuối cùng (last_hidden_state)
# Kích thước tensor: [batch_size, sequence_length, hidden_size]
hidden_states_1 = outputs_1.last_hidden_state
hidden_states_2 = outputs_2.last_hidden_state

print(f"Kích thước tensor đầu ra của câu 1: {hidden_states_1.shape}")
# Kích thước sẽ là [1, 10, 768] vì câu có 10 tokens (bao gồm cả [CLS] và [SEP]), mỗi token là 1 vector 768 chiều.

# Tìm vị trí index của từ "bank" trong mỗi câu
index_bank_1 = (inputs_1.input_ids[0] == tokenizer.vocab['bank']).nonzero(as_tuple=True)[0][0]
index_bank_2 = (inputs_2.input_ids[0] == tokenizer.vocab['bank']).nonzero(as_tuple=True)[0][0]

# Trích xuất vector 768 chiều của từ "bank" trong từng ngữ cảnh
vector_bank_1 = hidden_states_1[0, index_bank_1]
vector_bank_2 = hidden_states_2[0, index_bank_2]

# Tính độ tương đồng cosine giữa 2 vector của cùng một từ "bank"
cos = torch.nn.CosineSimilarity(dim=0)
similarity = cos(vector_bank_1, vector_bank_2)

print(f"Độ tương đồng cosine giữa 'bank' (ngân hàng) và 'bank' (bờ sông): {similarity.item():.4f}")
# Kết quả cosine similarity sẽ thấp, chứng minh rằng BERT hiểu "bank" ở 2 câu này mang 2 nghĩa hoàn toàn khác nhau.


--- 3. TRÍCH XUẤT CONTEXTUAL EMBEDDINGS ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Kích thước tensor đầu ra của câu 1: torch.Size([1, 11, 768])
Độ tương đồng cosine giữa 'bank' (ngân hàng) và 'bank' (bờ sông): 0.5105
